In [0]:
from pyspark.sql.functions import (
    col, current_timestamp, lit, row_number, trim, upper, to_timestamp, to_date
)
from pyspark.sql.window import Window
from delta.tables import DeltaTable

dbutils.widgets.text("catalog", "Databricks_prod")
CATALOG = dbutils.widgets.get("catalog")
BRONZE_SCHEMA = "bronze"
SILVER_SCHEMA = "silver"

In [0]:
spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SILVER_SCHEMA}")

In [0]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {CATALOG}.{SILVER_SCHEMA}._pipeline_watermark (
        table_name STRING,
        last_run_time TIMESTAMP,
        rows_processed BIGINT,
        run_status STRING
    ) USING DELTA
""")

print(f"Catalog: {CATALOG} | Bronze: {BRONZE_SCHEMA} | Silver: {SILVER_SCHEMA}")

In [0]:
TABLE_CONFIG = {
    "silver_customers": {
        "bronze_table": "customers",
        "primary_key": "customer_id",
        "order_by": "updated_at",
        "columns": {
            "customer_id": "int",
            "name":        "string",
            "city":        "string",
            "state":       "string",
            "signup_date": "date",
            "created_at":  "timestamp",
            "updated_at":  "timestamp",
        },
        "trim_cols":  ["name", "city", "state"],
        "upper_cols": ["state"],
    },
    "silver_products": {
        "bronze_table": "products",
        "primary_key": "product_id",
        "order_by": "updated_at",
        "columns": {
            "product_id":   "int",
            "product_name": "string",
            "category":     "string",
            "price":        "decimal(12,2)",
            "created_at":   "timestamp",
            "updated_at":   "timestamp",
        },
        "trim_cols":  ["product_name", "category"],
        "upper_cols": [],
    },
    "silver_orders": {
        "bronze_table": "order",
        "primary_key": "order_id",
        "order_by": "updated_at",
        "columns": {
            "order_id":     "int",
            "customer_id":  "int",
            "order_date":   "date",
            "total_amount": "decimal(12,2)",
            "order_status": "string",
            "created_at":   "timestamp",
            "updated_at":   "timestamp",
        },
        "trim_cols":  ["order_status"],
        "upper_cols": [],
    },
    "silver_order_items": {
        "bronze_table": "orderitems",
        "primary_key": "order_item_id",
        "order_by": "updated_at",
        "columns": {
            "order_item_id": "int",
            "order_id":      "int",
            "product_id":    "int",
            "quantity":      "int",
            "price":         "decimal(12,2)",
            "created_at":    "timestamp",
            "updated_at":    "timestamp",
        },
        "trim_cols":  [],
        "upper_cols": [],
    },
}


In [0]:
print(f"Registered {len(TABLE_CONFIG)} tables:")
for t in TABLE_CONFIG:
    print(f"  {TABLE_CONFIG[t]['bronze_table']} → {t}")

In [0]:
# ---- 1. INCREMENTAL READ ----
def read_incremental(bronze_table, last_run_time):
    """Read full or incremental based on watermark."""
    df = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.{bronze_table}")
    
    if last_run_time is None:
        print(f"  Mode: FULL LOAD")
        return df, "full"
    
    df_inc = df.filter(
        (col("created_at") > lit(last_run_time)) |
        (col("updated_at") > lit(last_run_time))
    )
    cnt = df_inc.count()
    print(f"  Mode: INCREMENTAL | Changed rows: {cnt:,}")
    return df_inc, "incremental"


# ---- 2. GENERIC CLEAN (metadata-driven) ----
def clean(df, config):
    """
    Apply type casting, trimming, and uppercasing based on config.
    Works for ANY table — just reads the config dictionary.
    """
    # Select only the columns defined in config (drop bronze audit cols)
    df_clean = df.select([c for c in config["columns"].keys()])
    
    # Type cast each column
    for col_name, col_type in config["columns"].items():
        if col_type == "date":
            df_clean = df_clean.withColumn(col_name, to_date(col(col_name)))
        elif col_type == "timestamp":
            df_clean = df_clean.withColumn(col_name, to_timestamp(col(col_name)))
        else:
            df_clean = df_clean.withColumn(col_name, col(col_name).cast(col_type))
    
    # Trim string columns
    for c in config.get("trim_cols", []):
        df_clean = df_clean.withColumn(c, trim(col(c)))
    
    # Uppercase columns
    for c in config.get("upper_cols", []):
        df_clean = df_clean.withColumn(c, upper(col(c)))
    
    # Add silver audit column
    df_clean = df_clean.withColumn("_silver_processed_at", current_timestamp())
    
    return df_clean


In [0]:

def deduplicate(df, primary_key, order_by_col):
    """Keep only the latest record per primary key."""
    window = Window.partitionBy(primary_key).orderBy(col(order_by_col).desc())
    return (df
            .withColumn("_rn", row_number().over(window))
            .filter(col("_rn") == 1)
            .drop("_rn"))


# ---- 4. GENERIC MERGE (UPSERT) ----
def merge_to_silver(df, target_table, primary_key, load_mode):
    """Full load or MERGE into silver Delta table."""
    if load_mode == "full" or not spark.catalog.tableExists(target_table):
        print(f"  Writing full load → {target_table}")
        (df.write
         .format("delta")
         .mode("overwrite")
         .option("overwriteSchema", "true")
         .saveAsTable(target_table))
    else:
        print(f"  Merging → {target_table}")
        delta_target = DeltaTable.forName(spark, target_table)
        (delta_target.alias("t")
         .merge(df.alias("s"), f"t.{primary_key} = s.{primary_key}")
         .whenMatchedUpdateAll()
         .whenNotMatchedInsertAll()
         .execute())


# ---- 5. WATERMARK HELPERS ----
def get_last_run_time(table_name):
    try:
        return spark.sql(f"""
            SELECT MAX(last_run_time) as lr 
            FROM {CATALOG}.{SILVER_SCHEMA}._pipeline_watermark
            WHERE table_name = '{table_name}' AND run_status = 'SUCCESS'
        """).collect()[0]["lr"]
    except Exception:
        return None

def update_watermark(table_name, rows):
    spark.sql(f"""
        INSERT INTO {CATALOG}.{SILVER_SCHEMA}._pipeline_watermark
        VALUES ('{table_name}', current_timestamp(), {rows}, 'SUCCESS')
    """)

In [0]:
results = []

for table_name, config in TABLE_CONFIG.items():
    print(f"\n{'='*60}")
    print(f"  Processing: {table_name}")
    print(f"{'='*60}")
    
    target = f"{CATALOG}.{SILVER_SCHEMA}.{table_name}"
    
    # Step 1: Read (incremental or full)
    last_run = get_last_run_time(table_name)
    df_raw, load_mode = read_incremental(config["bronze_table"], last_run)
    
    # Step 2: Clean (generic - driven by config)
    df_cleaned = clean(df_raw, config)
    
    # Step 3: Deduplicate (generic)
    df_deduped = deduplicate(df_cleaned, config["primary_key"], config["order_by"])
    
    row_count = df_deduped.count()
    print(f"  Clean & deduped rows: {row_count:,}")
    
    # Step 4: Merge (generic)
    merge_to_silver(df_deduped, target, config["primary_key"], load_mode)
    
    # Step 5: Update watermark
    update_watermark(table_name, row_count)
    
    final_count = spark.table(target).count()
    results.append({"table": table_name, "processed": row_count, "total": final_count})
    print(f"  Silver total: {final_count:,} rows")


In [0]:
print(f"\n{'='*70}")
print(f"  SILVER LAYER SUMMARY")
print(f"{'='*70}")
print(f"  {'Table':<30s} | {'Processed':>10s} | {'Total':>10s}")
print(f"  {'-'*30}-+-{'-'*10}-+-{'-'*10}")
for r in results:
    print(f"  {r['table']:<30s} | {r['processed']:>10,} | {r['total']:>10,}")
print(f"{'='*70}")

In [0]:
from pyspark.sql.functions import broadcast

df_orders = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.silver_orders")
df_items  = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.silver_order_items")
df_cust   = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.silver_customers")
df_prod   = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.silver_products")

In [0]:
df_enriched = (df_items.alias("oi")
    .join(df_orders.alias("o"), "order_id", "inner")
    .select(
        col("oi.order_item_id"), col("oi.order_id"), col("o.customer_id"),
        col("oi.product_id"), col("o.order_date"), col("o.order_status"),
        col("oi.quantity"), col("oi.price").alias("unit_price"),
        col("o.total_amount").alias("order_total"),
        col("oi.created_at"), col("oi.updated_at"),
    )
)

# + customers (BROADCAST - 3.9 MB, small)
df_enriched = (df_enriched
    .join(broadcast(df_cust), "customer_id", "left")
    .select(
        df_enriched["*"],
        col("name").alias("customer_name"),
        col("city").alias("customer_city"),
        col("state").alias("customer_state"),
    )
)

# + products (BROADCAST - 361 KB, tiny)
df_enriched = (df_enriched
    .join(broadcast(df_prod), "product_id", "left")
    .select(
        df_enriched["*"],
        col("product_name"),
        col("category").alias("product_category"),
        col("price").alias("product_list_price"),
    )
)

df_enriched = df_enriched.withColumn("_silver_processed_at", current_timestamp())

# Write enriched table
target = f"{CATALOG}.{SILVER_SCHEMA}.silver_orders_enriched"
pk = "order_item_id"
df_final = deduplicate(df_enriched, pk, "updated_at")

merge_to_silver(df_final, target, pk, 
                "full" if not spark.catalog.tableExists(target) else "incremental")
update_watermark("silver_orders_enriched", df_final.count())

print(f"\nEnriched orders: {spark.table(target).count():,} rows")
